# 05 — 变分推断与 VAE
第 04 章通过网格积分直接 MLE；隐变量维度增加后，积分成本随维度快速增长。这里不改变生成模型，也不改变最大化观测似然的最终目标，而是使用一个可计算的下界替代直接积分。

生成模型仍然为 $p(x)=N(0,1)$、$p_\theta(y|x)=N(f_\theta(x),\sigma^2I)$。新增 $q_\phi(x|y)=N(m_\phi(y),s_\phi(y)^2)$，由编码器定义。它不是模型的真实后验。

$$\log p_\theta(y)=\underbrace{\mathbb E_q[\log p_\theta(y|x)]-D_{KL}(q\|p(x))}_{ELBO}+D_{KL}(q\|p_\theta(x|y)).$$
先固定生成模型，仅优化近似后验（变分推断）；再同时优化两个网络（VAE）。若上一章已运行，读取其生成模型。否则使用下面的同样的网格训练构建相同模型。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 复用第 04 章生成模型与数值积分
网格积分仍用于低维验证和准备一个固定生成模型，但不再作为 VAE 联合训练的损失。新增的编码器输出近似后验的均值和标准差，而非生成模型的参数。

In [ ]:
import torch
from torch import nn
from pathlib import Path
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float32)
sigma = .45
# 与第 04 章完全相同的生成网络。
def make_decoder():
    return nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(), nn.Linear(32,2))
decoder = make_decoder()

# 对每个观测，枚举隐变量网格。每行是一组联合对数密度。
def log_joint(obs, grid):
    means = decoder(grid)
    lp = torch.distributions.Normal(0., 1.).log_prob(grid[:, 0])
    rows = []
    for i in range(len(obs)):
        # 同一个观测在所有网格点对应的二维条件分布下的密度。
        # 将观测复制到每个网格点。
        repeated_obs = obs[i].repeat(len(grid), 1)
        lc = torch.distributions.Normal(means, sigma).log_prob(repeated_obs).sum(dim=1)
        rows.append(lc + lp)
    return torch.stack(rows)

def grid_log_marginal(obs, grid):
    joint = log_joint(obs, grid)
    dx = grid[1, 0] - grid[0, 0]
    values = []
    for i in range(len(obs)):
        values.append(torch.logsumexp(joint[i], dim=0) + torch.log(dx))
    return torch.stack(values)

# 优先沿用上一章的参数；没有文件才重新训练相同模型。
if Path('decoder_grid.pt').exists():
    decoder.load_state_dict(torch.load('decoder_grid.pt', weights_only=True))
else:
    g = torch.linspace(-6,6,241).reshape(-1,1)
    opt = torch.optim.Adam(decoder.parameters(), lr=.01)
    for _ in range(350):
        obs = Y[torch.randint(len(Y), (128,))]
        loss = -grid_log_marginal(obs,g).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

# 新增编码器：输入二维 y，输出一维隐变量的均值和 log 标准差。
encoder = nn.Sequential(nn.Linear(2,32), nn.Tanh(), nn.Linear(32,2))
def posterior_params(obs):
    output = encoder(obs)
    # 第 0 列为均值，第 1 列为 log 标准差；限制范围以保证数值稳定。
    return output[:,0], output[:,1].clamp(-5,2)

def elbo(obs, n_samples=8):
    m, log_s = posterior_params(obs)
    s = log_s.exp()  # 标准差为正
    reconstruction = torch.zeros(len(obs), dtype=obs.dtype)
    # 每次为每个样本采一个隐变量；同形状张量逐元素计算。
    for j in range(n_samples):
        # 先采标准正态，再变换为 q(x|y) 的样本，保留对参数的梯度。
        epsilon = torch.randn(len(obs))
        x = m + s*epsilon
        # 沿用第 04 章的条件均值模型。
        mean = decoder(x.reshape(-1,1))
        log_conditional = torch.distributions.Normal(mean,sigma).log_prob(obs).sum(dim=1)
        reconstruction = reconstruction + log_conditional
    reconstruction = reconstruction / n_samples
    # 一维高斯近似后验与标准正态先验的解析 KL。
    kl = .5*(m.square()+s.square()-1-2*log_s)
    return (reconstruction-kl).mean()

# 第一阶段：固定生成模型，只学习近似后验。
for p in decoder.parameters():
    p.requires_grad_(False)
opt = torch.optim.Adam(encoder.parameters(),lr=.005)
for _ in range(500):
    obs = Y[torch.randint(len(Y),(128,))]
    loss = -elbo(obs)
    opt.zero_grad()
    loss.backward()
    opt.step()

In [ ]:
# 沿用第 04 章的网格后验，作为低维参考；不参与训练。
def compare_posterior(index):
    with torch.no_grad():
        obs = Y[index:index+1]
        g = torch.linspace(-6,6,1001).reshape(-1,1)
        lj = log_joint(obs,g)[0]
        log_z = grid_log_marginal(obs,g)[0]
        m, log_s = posterior_params(obs)
        q = torch.distributions.Normal(m[0],log_s[0].exp())
        model_values = []
        q_values = []
        for k in range(len(g)):
            model_values.append(torch.exp(lj[k]-log_z).item())
            q_values.append(q.log_prob(g[k,0]).exp().item())
        plt.plot(g[:,0],model_values,label='model posterior (grid)')
        plt.plot(g[:,0],q_values,label='q')
        plt.legend()
        plt.show()
        print('log evidence:',log_z.item(),'MC ELBO:',elbo(obs,1000).item())
compare_posterior(0)
compare_posterior(12)

固定生成模型时，最大化 ELBO 等价于最小化 q 与真实后验之间的 KL 散度。高斯 q 不一定能表示真实后验的多个峰；编码器也不一定能对每个样本达到最优。蒙特卡洛估计的 ELBO 可能因采样误差偶尔超过数值似然。

In [ ]:
# 第二阶段：重新初始化两个网络，联合最大化 ELBO。
# 不再调用上一章的网格积分计算训练损失。
decoder=make_decoder()
encoder=nn.Sequential(nn.Linear(2,32),nn.Tanh(),nn.Linear(32,2))
opt=torch.optim.Adam(list(decoder.parameters())+list(encoder.parameters()),lr=.005)
history=[]
for _ in range(1000):
    obs=Y[torch.randint(len(Y),(128,))]
    loss = -elbo(obs)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(-loss.item())
plot_history(history, 'MC ELBO')
compare_posterior(0)
# 生成数据仍然只使用先验和解码器，编码器不参与。
with torch.no_grad():
    samples = []
    for i in range(len(y)):
        x = torch.randn(1)
        samples.append((decoder(x) + sigma*torch.randn(2)).numpy())
    compare(y, np.array(samples))

改变隐变量维度将需要同时调整先验、编码器和解码器；训练时的采样仍可扩展，不需要指数增长的网格。VAE 的生成质量受模型、后验近似和优化影响，不保证拟合优于直接 MLE。